In [1]:
import os
from pathlib import Path
import datetime

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl 
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet, ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler


import kaggle_evaluation.default_inference_server

In [2]:
import numpy as np
import pandas as pd
import pandas.api.types

MIN_INVESTMENT = 0
MAX_INVESTMENT = 2


class ParticipantVisibleError(Exception):
    pass


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Calculates a custom evaluation metric (volatility-adjusted Sharpe ratio).

    This metric penalizes strategies that take on significantly more volatility
    than the underlying market.

    Returns:
        float: The calculated adjusted Sharpe ratio.
    """

    if not pandas.api.types.is_numeric_dtype(submission['prediction']):
        raise ParticipantVisibleError('Predictions must be numeric')

    solution = solution
    solution['position'] = submission['prediction']

    if solution['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
    if solution['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].min()} below minimum of {MIN_INVESTMENT}')

    solution['strategy_returns'] = solution['risk_free_rate'] * (1 - solution['position']) + solution['position'] * solution['forward_returns']

    # Calculate strategy's Sharpe ratio
    strategy_excess_returns = solution['strategy_returns'] - solution['risk_free_rate']
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solution)) - 1
    strategy_std = solution['strategy_returns'].std()

    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ParticipantVisibleError('Division by zero, strategy std is zero')
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)

    # Calculate market return and volatility
    market_excess_returns = solution['forward_returns'] - solution['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solution)) - 1
    market_std = solution['forward_returns'].std()

    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)

    if market_volatility == 0:
        raise ParticipantVisibleError('Division by zero, market std is zero')

    # Calculate the volatility penalty
    excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
    vol_penalty = 1 + excess_vol

    # Calculate the return penalty
    return_gap = max(
        0,
        (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr,
    )
    return_penalty = 1 + (return_gap**2) / 100

    # Adjust the Sharpe ratio by the volatility and return penalty
    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)

In [3]:
# ============ PATHS ============
DATA_PATH: Path = Path('/kaggle/input/hull-tactical-market-prediction/')

# ============ RETURNS TO SIGNAL CONFIGS ============
MIN_SIGNAL: float = 0.0                         # Minimum value for the daily signal 
MAX_SIGNAL: float = 2.0                         # Maximum value for the daily signal 
SIGNAL_MULTIPLIER: float = 49.0                # Multiplier of the OLS market forward excess returns predictions to signal 
# ============ MODEL CONFIGS ============
CV: int = 10                                    # Number of cross validation folds in the model fitting
L1_RATIO: float = 0.40                          # ElasticNet mixing parameter
ALPHAS: np.ndarray = np.logspace(-4, 2, 100)    # Constant that multiplies the penalty terms
MAX_ITER: int = 1000000                         # The maximum number of iterations

NON_FEATURE_COLS = [
    'forward_returns',
    'risk_free_rate',
    'lagged_risk_free_rate',
    'lagged_market_forward_excess_returns',
    'is_scored',
]

@dataclass
class DatasetOutput:
    X_train : pl.DataFrame 
    X_test: pl.DataFrame
    y_train: pl.Series
    y_test: pl.Series
    scaler: any

@dataclass 
class ElasticNetParameters:
    l1_ratio: any
    cv: int
    alphas: np.ndarray
    max_iter: int
    
    def __post_init__(self): 
        if self.l1_ratio < 0 or self.l1_ratio > 1: 
            raise ValueError("Wrong initializing value for ElasticNet l1_ratio")
        
@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier: float = SIGNAL_MULTIPLIER
    min_signal: float = MIN_SIGNAL
    max_signal: float = MAX_SIGNAL

ret_signal_params = RetToSignalParameters(
    signal_multiplier = SIGNAL_MULTIPLIER,
    min_signal = MIN_SIGNAL,
    max_signal = MAX_SIGNAL
)

In [4]:
def load_trainset() -> pl.DataFrame:
    """
    Loads and preprocesses the training dataset.

    Returns:
        pl.DataFrame: The preprocessed training DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "train.csv")
        .rename({'market_forward_excess_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
        .head(-10)
    )

def load_testset() -> pl.DataFrame:
    """
    Loads and preprocesses the testing dataset.

    Returns:
        pl.DataFrame: The preprocessed testing DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "test.csv")
        .rename({'lagged_forward_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
    )

def create_example_dataset(df: pl.DataFrame) -> pl.DataFrame:
    """
    Creates new features and cleans a DataFrame.

    Args:
        df (pl.DataFrame): The input Polars DataFrame.

    Returns:
        pl.DataFrame: The DataFrame with new features, selected columns, and no null values.
    """
    df = df.sort('date_id')
    
    ewm_cols = [col for col in df.columns if col not in ['date_id', 'target'] + NON_FEATURE_COLS]
    for c in ['U1', 'U2', 'mom_I2', 'vol_I2']:
        ewm_cols.append(c)
    
    df = df.with_columns(
        (pl.col("I2") - pl.col("I1")).alias("U1"),
        (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3)).alias("U2")
    )

    df = df.with_columns(
        (pl.col('I2').diff().ewm_mean(com=3)).alias('mom_I2'),
        (pl.col('I2').diff().abs().ewm_mean(com=5)).alias('vol_I2')
    )

    df = df.with_columns([
        pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5))
        for col in ewm_cols
    ])

    df = df.with_columns([
        pl.col(col).fill_null(0.0)
        for col in ewm_cols
    ])
    
    return df

def join_train_test_dataframes(train: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    """
    Joins two dataframes by common columns and concatenates them vertically.

    Args:
        train (pl.DataFrame): The training DataFrame.
        test (pl.DataFrame): The testing DataFrame.

    Returns:
        pl.DataFrame: A single DataFrame with vertically stacked data from common columns.
    """
    common_columns: list[str] = [col for col in train.columns if col in test.columns]
    
    return pl.concat([train.select(common_columns), test.select(common_columns)], how="vertical")

def split_dataset(train: pl.DataFrame, test: pl.DataFrame, features: list[str]) -> DatasetOutput: 
    """
    Splits the data into features (X) and target (y), and scales the features.

    Args:
        train (pl.DataFrame): The processed training DataFrame.
        test (pl.DataFrame): The processed testing DataFrame.
        features (list[str]): List of features to used in model. 

    Returns:
        DatasetOutput: A dataclass containing the scaled feature sets, target series, and the fitted scaler.
    """
    X_train = train.drop(['date_id','target']) 
    y_train = train.get_column('target')
    X_test = test.drop(['date_id','target']) 
    y_test = test.get_column('target')
    
    scaler = StandardScaler() 
    # scaler = RobustScaler()
    X_train_scaled_np = scaler.fit_transform(X_train)
    X_train_scaled_np = np.nan_to_num(X_train_scaled_np, nan=0.0)
    X_train = pl.from_numpy(X_train_scaled_np, schema=features)
    
    X_test_scaled_np = scaler.transform(X_test)
    X_test_scaled_np = np.nan_to_num(X_test_scaled_np, nan=0.0)
    X_test = pl.from_numpy(X_test_scaled_np, schema=features)
    
    
    return DatasetOutput(
        X_train = X_train,
        y_train = y_train, 
        X_test = X_test, 
        y_test = y_test,
        scaler = scaler
    )

def convert_ret_to_signal(
    ret_arr: np.ndarray,
    params
) -> np.ndarray:
    """
    Converts raw model predictions (expected returns) into a trading signal.

    Args:
        ret_arr (np.ndarray): The array of predicted returns.
        params (RetToSignalParameters): Parameters for scaling and clipping the signal.

    Returns:
        np.ndarray: The resulting trading signal, clipped between min and max values.
    """
    return np.clip(
        ret_arr * params.signal_multiplier + 1, params.min_signal, params.max_signal
    )

In [5]:
train: pl.DataFrame = load_trainset()
test: pl.DataFrame = load_testset() 

print(train.tail(3)) 
print(test.head(3))
print(train.shape, test.shape)

train = create_example_dataset(train)
test = create_example_dataset(test)

train = train.drop([col for col in NON_FEATURE_COLS if col in train.columns])
test  = test.drop([col for col in NON_FEATURE_COLS if col in test.columns])

FEATURES: list[str] = [col for col in train.columns if col not in ['date_id', 'target']]

print(train.shape, test.shape, len(FEATURES))
dataset: DatasetOutput = split_dataset(train=train, test=test, features=FEATURES) 

X_train: pl.DataFrame = dataset.X_train
X_test: pl.DataFrame = dataset.X_test
y_train: pl.DataFrame = dataset.y_train
y_test: pl.DataFrame = dataset.y_test
scaler: StandardScaler = dataset.scaler 

shape: (3, 98)
┌─────────┬─────┬─────┬─────┬───┬───────────┬─────────────────┬────────────────┬───────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9        ┆ forward_returns ┆ risk_free_rate ┆ target    │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---             ┆ ---            ┆ ---       │
│ i64     ┆ f64 ┆ f64 ┆ f64 ┆   ┆ f64       ┆ f64             ┆ f64            ┆ f64       │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪═════════════════╪════════════════╪═══════════╡
│ 9008    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.530228 ┆ -0.002897       ┆ 0.0001525      ┆ -0.003362 │
│ 9009    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.512769 ┆ -0.027028       ┆ 0.000153       ┆ -0.027493 │
│ 9010    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.015503 ┆ 0.015344        ┆ 0.000153       ┆ 0.014879  │
└─────────┴─────┴─────┴─────┴───┴───────────┴─────────────────┴────────────────┴───────────┘
shape: (3, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬───────────┬─────────────────────┬────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … 

In [6]:
dates = train['date_id'].unique().sort()
cut = int(len(dates) * 0.8)

train_dates = dates[:cut]
valid_dates = dates[cut:]

train_sub = train.filter(pl.col('date_id').is_in(train_dates))
valid_sub = train.filter(pl.col('date_id').is_in(valid_dates))

dataset_sub = split_dataset(
    train=train_sub,
    test=valid_sub,
    features=FEATURES
)

X_train_sub = dataset_sub.X_train
y_train_sub = dataset_sub.y_train
X_valid_sub = dataset_sub.X_test
y_valid_sub = dataset_sub.y_test
scaler_sub = dataset_sub.scaler

In [7]:
tscv = TimeSeriesSplit(n_splits=10)

# model_cv = ElasticNetCV(
#     l1_ratio=L1_RATIO,
#     cv=tscv,
#     alphas=ALPHAS,
#     max_iter=MAX_ITER
# )
# model_cv.fit(X_train_sub, y_train_sub)

best_alpha = 7.196856730011514e-05
model = ElasticNet(alpha=best_alpha, l1_ratio=L1_RATIO) 
model.fit(X_train_sub, y_train_sub)

ElasticNet(alpha=7.196856730011514e-05, l1_ratio=0.4)

# Top-K Correlated Feature grid search

In [8]:
# train_df = train_sub.to_pandas()

# corrs = {}
# for col in FEATURES:
#     corr = train_df[col].corr(train_df['target'])
#     if pd.isna(corr):
#         corr = 0
#     corrs[col] = abs(corr)

# sorted_feat_by_corrs = sorted(corrs.items(), key=lambda x: x[1], reverse=True)
# print(sorted_feat_by_corrs)
# feature_counts = [10, 20, 30, 40, 50, 60, 70, 80, 90, 96]

# for k in feature_counts:
#     chosen = [name for name, _ in sorted_feat_by_corrs[:k]]
#     print(len(chosen))
#     train_k = train_sub.select(['date_id', 'target'] + chosen)
#     valid_k = valid_sub.select(['date_id', 'target'] + chosen)
    
#     dataset_k = split_dataset(train_k, valid_k, chosen)
    
#     X_train_k = dataset_k.X_train
#     y_train_k = dataset_k.y_train
#     X_valid_k = dataset_k.X_test
#     y_valid_k = dataset_k.y_test

#     model = ElasticNet(alpha=best_alpha, l1_ratio=L1_RATIO) 
#     model.fit(X_train_k, y_train_k)

#     raw_preds = model.predict(X_valid_k)
#     raw_train = load_trainset().to_pandas()
#     processed_valid = valid_sub.to_pandas()
#     sig = convert_ret_to_signal(raw_preds, ret_signal_params)
    
#     solution = processed_valid[['date_id']].merge(
#             raw_train,
#             on='date_id',
#             how='left'
#         )
#     submission = pd.DataFrame({
#             'id': solution.index,
#             'prediction': sig
#         })
#     sc = score(solution, submission, 'id')
#     print(sc)

# Alpha grid search

In [9]:
# alpha_grid = np.logspace(-5, -2, 50)

# best_alpha = None
# best_score = -1e9

# raw_train = load_trainset().to_pandas()
# processed_valid = valid_sub.to_pandas()

# solution_valid = processed_valid[['date_id']].merge(
#     raw_train,
#     on='date_id',
#     how='left'
# )

# for alpha in alpha_grid:
#     model = ElasticNet(alpha=alpha, l1_ratio=L1_RATIO, max_iter=MAX_ITER)
#     model.fit(X_train_sub, y_train_sub)

#     raw_preds = model.predict(X_valid_sub)
#     sig = convert_ret_to_signal(raw_preds, ret_signal_params)

#     submission = pd.DataFrame({
#         'id': solution.index,
#         'prediction': sig
#     })

#     sc = score(solution, submission, 'id')
#     # print(f'Alpha: {alpha:.10f} | Score: {sc:.10f}')

#     if sc > best_score:
#         best_score = sc
#         best_alpha = alpha

# print(f'Best Alpha: {best_alpha} | Best Score: {best_score}')

# L1_ratio grid search

In [10]:
# raw_train = load_trainset().to_pandas()
# processed_valid = valid_sub.to_pandas()
# solution = processed_valid[['date_id']].merge(
#         raw_train,
#         on='date_id',
#         how='left'
#     )

# l1_ratios = np.linspace(0.1, 0.9, 30)

# best_ratio = 0
# best_score = 0

# for ratio in l1_ratios:
#     model_cv = ElasticNetCV(
#         l1_ratio=ratio,
#         cv=tscv,
#         alphas=ALPHAS,
#         max_iter=MAX_ITER
#     )
#     model_cv.fit(X_train_sub, y_train_sub)
    
#     model = ElasticNet(alpha=model_cv.alpha_, l1_ratio=ratio)
#     model.fit(X_train_sub, y_train_sub)
    
#     raw_preds = model.predict(X_valid_sub)
#     sig = convert_ret_to_signal(raw_preds, ret_signal_params)
#     submission = pd.DataFrame({
#             'id': solution.index,
#             'prediction': sig
#         })
#     sc = score(solution, submission, 'id')
#     print(f'Score: {sc} | Ratio: {ratio}')

#     if sc > best_score:
#         best_score = sc
#         best_ratio = ratio

# print(f'Best Ratio: {best_ratio} | Score: {best_score}')
        

# SIGNAL_MULTIPLIER grid search

In [11]:
# raw_preds = model.predict(X_valid_sub)
# multipliers = np.linspace(0, 1000, num=50)

# best_mp = 0
# best_score = 0

# for mp in multipliers:
#     mp_params = RetToSignalParameters(
#         signal_multiplier = mp
#     )
#     sig = convert_ret_to_signal(raw_preds, mp_params)
#     submission = pd.DataFrame({
#         'id': solution.index,
#         'prediction': sig
#     })
    
#     sc = score(solution, submission, 'id')
#     print(f'Score: {sc} for MP: {mp}')

#     if sc > best_score:
#         best_mp = mp
#         best_score = sc

# print(f'Best MP: {best_mp} | Score: {best_score}')

In [12]:
raw_preds = model.predict(X_valid_sub)
raw_train = load_trainset().to_pandas()
processed_valid = valid_sub.to_pandas()
sig = convert_ret_to_signal(raw_preds, ret_signal_params)

solution = processed_valid[['date_id']].merge(
        raw_train,
        on='date_id',
        how='left'
    )
submission = pd.DataFrame({
        'id': solution.index,
        'prediction': sig
    })
sc = score(solution, submission, 'id')
print(sc)

0.6559130928243201


In [13]:
model = ElasticNet(alpha=best_alpha, l1_ratio=L1_RATIO)
model.fit(X_train, y_train)

ElasticNet(alpha=7.196856730011514e-05, l1_ratio=0.4)

In [14]:
def predict(test: pl.DataFrame) -> float:
    test = test.rename({'lagged_forward_returns':'target'})
    df: pl.DataFrame = create_example_dataset(test)
    X_test: pl.DataFrame = df.select(FEATURES)
    X_test_scaled_np: np.ndarray = scaler.transform(X_test)
    X_test: pl.DataFrame = pl.from_numpy(X_test_scaled_np, schema=FEATURES)
    raw_pred: float = model.predict(X_test)[0]
    return convert_ret_to_signal(raw_pred, ret_signal_params)

In [15]:
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))